# LLM Evaluation Dashboard — Colab Walkthrough

This notebook demonstrates the evaluation pipeline standalone (no Streamlit UI needed): loading the benchmark dataset, running evaluations against Gemini models, comparing prompt versions and models, and analyzing latency, tokens, cost, and hallucination.

> Uses the **free tier** of the Gemini API. Get a key at https://aistudio.google.com/app/apikey

## 1. Install dependencies

In [ ]:
!pip install -q google-generativeai pandas plotly scikit-learn pydantic loguru

## 2. Clone/mount the project (or upload `utils/`, `models/`, `data/` manually)

In [ ]:
# If running from Google Drive or a cloned repo, adjust this path so the
# `utils` and `models` packages are importable.
import sys
PROJECT_ROOT = "."  # change to e.g. "/content/llm-evaluation-dashboard"
sys.path.append(PROJECT_ROOT)

## 3. Configure your Gemini API key

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
os.environ["FREE_TIER_MODE"] = "true"

## 4. Load the benchmark dataset

In [ ]:
from utils.benchmark import load_benchmark, dataset_to_dataframe

benchmark_items = load_benchmark(f"{PROJECT_ROOT}/data/benchmark_dataset.csv")
dataset_to_dataframe(benchmark_items).head()

## 5. Define prompt versions to compare

In [ ]:
from models.evaluation import PromptVersion

prompt_versions = [
    PromptVersion(
        name="baseline", version="1.0",
        template="Answer the following question accurately and concisely.\n\nQuestion: {question}\nAnswer:"
    ),
    PromptVersion(
        name="chain_of_thought", version="1.1",
        template="Think step by step, then give a concise final answer.\n\nQuestion: {question}\nAnswer:"
    ),
]

## 6. Run the evaluation sweep across models and prompt versions

In [ ]:
from utils.evaluator import LLMEvaluator

MODELS = ["gemini-1.5-flash", "gemini-1.5-flash-8b"]
api_key = os.environ["GOOGLE_API_KEY"]

all_results = []
for model_name in MODELS:
    evaluator = LLMEvaluator(api_key=api_key, model_name=model_name)
    for prompt_version in prompt_versions:
        for item in benchmark_items:
            result = evaluator.evaluate(item, prompt_version)
            all_results.append(result)
            print(f"[{model_name} | {prompt_version.label}] {item.question[:40]:40s} -> "
                  f"acc={result.overall_accuracy:.1f} halluc={result.hallucination_score:.1f} "
                  f"latency={result.latency_ms:.0f}ms")

## 7. Build a results DataFrame

In [ ]:
from utils.storage import build_summary

df = build_summary(all_results)
df.head()

## 8. Compare models and prompt versions

In [ ]:
df.groupby("model_name")[["overall_accuracy", "hallucination_score", "latency_ms", "cost_usd"]].mean()

In [ ]:
df.groupby("prompt_version")[["overall_accuracy", "hallucination_score"]].mean()

## 9. Visualize latency, cost, tokens, and hallucination trend

In [ ]:
from utils.charts import latency_chart, cost_chart, token_usage_chart, accuracy_by_prompt_chart, hallucination_trend_chart

latency_chart(df).show()
cost_chart(df).show()
token_usage_chart(df).show()
accuracy_by_prompt_chart(df).show()
hallucination_trend_chart(df).show()

## 10. Inspect hallucination-flagged responses

In [ ]:
flagged = df[df["hallucination_score"] >= 40]
flagged[["model_name", "prompt_version", "question", "hallucination_score", "hallucination_flags"]]

## 11. Export a report

In [ ]:
from utils.storage import export_csv, export_json

export_csv(df, "results/evaluation_results.csv")
export_json(df, "reports/evaluation_report.json")
print("Report exported.")